# 03 — Finetune Qwen3-TTS-1.7B-Base (voz do Pedro) · LoRA patchado

Candidato **#1 da trilha A** (Apache-2.0, pt nativo, ~97ms streaming). Receita
verificada em 2026-06-10: script oficial `sft_12hz.py` tem 2 bugs conhecidos
(`text_projection` faltante + double label-shift → fala acelera a cada época);
usamos o repo **cheeweijie/qwen3-tts-lora-finetuning** que aplica os patches.

**GPU: L4 (24GB) ou A100 — NÃO use T4** (Turing sem bf16; nenhuma fonte valida 1.7B em T4).
**Base**: `Qwen3-TTS-12Hz-1.7B-Base` (o card do CustomVoice não documenta finetune).
**Dataset**: texto CRU sem tags (`<animado>` confunde o Qwen3 — emoção aqui vai via
`instruct` na inferência; tags ficam pro CSM/notebook 2). Mesmo `ref_audio` em todas as linhas.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
GH_TOKEN = userdata.get('GH_TOKEN')
!git clone https://{GH_TOKEN}@github.com/pedrocormann/TTS-ptbr.git /content/TTS-ptbr 2>/dev/null || (cd /content/TTS-ptbr && git pull)
%cd /content/TTS-ptbr
!mkdir -p data && ln -sfn /content/drive/MyDrive/TTS-ptbr-data/dataset_v1 data/dataset_v1

In [ ]:
import os; os.environ['HF_HUB_ENABLE_HF_TRANSFER']='1'
# qwen-tts pina transformers==4.57.3 e accelerate==1.12.0 — NAO atualizar
!pip install -U qwen-tts peft soundfile jiwer librosa hf_transfer
# repo oficial no commit que o patch espera + repo com os fixes
!git clone https://github.com/QwenLM/Qwen3-TTS.git /content/Qwen3-TTS
!cd /content/Qwen3-TTS && git checkout 0c6a7cbb6c8421a46332f8c2434c7825c4c855ef
!git clone https://github.com/cheeweijie/qwen3-tts-lora-finetuning.git /content/q3lora
!cd /content/q3lora && QWEN_DIR=/content/Qwen3-TTS bash scripts/apply_patches.sh

## 1. Dataset → JSONL {audio 24kHz, text cru, ref_audio fixo}

In [ ]:
import json, pathlib, re

ROOT = pathlib.Path('data/dataset_v1').resolve()
rows = [json.loads(l) for l in (ROOT/'train.jsonl').read_text(encoding='utf-8').splitlines() if l.strip()]
rows = [r for r in rows if r.get('kind') != 'paralinguistico']   # sem tags/eventos no Qwen3

# ref_audio: 1 clipe neutro limpo, 3-10s, o MESMO em todas as linhas (recomendação oficial)
neutros = [r for r in rows if r.get('style') == 'neutro' and 3.0 <= r.get('dur_s', 0) <= 10.0]
REF = str(ROOT / neutros[0]['audio'])
print('ref_audio:', REF)

strip_tags = lambda t: re.sub(r'<[^>]*>\s*', '', t).strip()
with open('/content/train_raw.jsonl', 'w', encoding='utf-8') as f:
    for r in rows:
        f.write(json.dumps({'audio': str(ROOT/r['audio']), 'text': strip_tags(r['text']),
                            'ref_audio': REF}, ensure_ascii=False) + '\n')
rows_val = [json.loads(l) for l in (ROOT/'val.jsonl').read_text(encoding='utf-8').splitlines() if l.strip()]
with open('/content/val_raw.jsonl', 'w', encoding='utf-8') as f:
    for r in rows_val:
        f.write(json.dumps({'audio': str(ROOT/r['audio']), 'text': strip_tags(r['text']),
                            'ref_audio': REF}, ensure_ascii=False) + '\n')
print(len(rows), 'train /', len(rows_val), 'val (áudio já é 24kHz mono PCM16 do export)')

In [ ]:
# codifica áudio → codes (uma vez; em dataset grande, processe em chunks — issue #5 memory leak)
!cd /content/Qwen3-TTS && python finetuning/prepare_data.py \
    --device cuda:0 \
    --tokenizer_model_path Qwen/Qwen3-TTS-Tokenizer-12Hz \
    --input_jsonl /content/train_raw.jsonl --output_jsonl /content/train_codes.jsonl
!cd /content/Qwen3-TTS && python finetuning/prepare_data.py \
    --device cuda:0 \
    --tokenizer_model_path Qwen/Qwen3-TTS-Tokenizer-12Hz \
    --input_jsonl /content/val_raw.jsonl --output_jsonl /content/val_codes.jsonl

## 2. Treino LoRA (patchado: r=16/α=32, lr 2e-6 — 2e-5 oficial gera ruído)

In [ ]:
!cd /content/q3lora && QWEN_DIR=/content/Qwen3-TTS ATTN_IMPL=sdpa \
    TRAIN_JSONL=/content/train_codes.jsonl VAL_JSONL=/content/val_codes.jsonl \
    OUTPUT_DIR=/content/output LR=2e-6 EPOCHS=5 \
    bash scripts/run_lora_train.sh
# sinais de problema: fala ACELERANDO a cada época = double label-shift (patch não aplicado);
# ruído com loss caindo = LR alto. 3-5 épocas bastam; >10 = voz robótica (overfit).

## 3. Inferência — sweep de LORA_SCALE (1.0 "over-steers"; faixa segura 0.25–0.35)

In [ ]:
import json, pathlib, subprocess, os
bench = [json.loads(l) for l in open('eval/benchmark_ptbr.jsonl', encoding='utf-8') if l.strip()]
CKPT = sorted(pathlib.Path('/content/output').glob('checkpoint-epoch-*'))[-1]

for scale in ['0.2', '0.3', '0.35', '0.5']:
    outdir = pathlib.Path('/content/TTS-ptbr').resolve() / f'gen_qwen3_s{scale}'; outdir.mkdir(exist_ok=True)
    for i, item in enumerate(bench):
        # env COMPLETO do Colab (env mínimo apagaria LD_LIBRARY_PATH/CUDA/HF_HOME) + sdpa
        # (flash-attn não está instalado; o script usa flash_attention_2 por default → trocar)
        subprocess.run(['bash', 'scripts/run_lora_infer.sh'], cwd='/content/q3lora', check=True,
            env=dict(os.environ, QWEN_DIR='/content/Qwen3-TTS',
                     BASE_MODEL='Qwen/Qwen3-TTS-12Hz-1.7B-Base', ATTN_IMPL='sdpa',
                     ADAPTER_DIR=str(CKPT), LORA_SCALE=scale, TEXT=item['text'],
                     OUT_WAV=str((outdir / f"{item.get('id', i):0>3}.wav").resolve())))
    print('scale', scale, 'ok →', outdir)

## 4. Eval por scale (gates F1: spk-sim ≥ 0.70 · WER ≤ 1.2× real) + escuta

In [ ]:
!pip -q install faster-whisper==1.1.0
!mkdir -p ref_pedro && python -c "
import json, shutil, pathlib
rows=[json.loads(l) for l in open('data/dataset_v1/train.jsonl', encoding='utf-8') if l.strip()]
neutros=[r for r in rows if r.get('style')=='neutro'][:20]
[shutil.copy(pathlib.Path('data/dataset_v1')/r['audio'], 'ref_pedro/') for r in neutros]"
for scale in ['0.2', '0.3', '0.35', '0.5']:
    print(f'===== LORA_SCALE {scale} =====')
    !python -m eval.wer_roundtrip --in-dir gen_qwen3_s{scale} --transcripts eval/benchmark_ptbr.jsonl --model medium --lang pt
    !python -m eval.speaker_sim --ref-dir ref_pedro --gen-dir gen_qwen3_s{scale}

## 5. Salvar no Drive + teste de `instruct` (emoção)
> `instruct` é capacidade do CustomVoice; num Base finetunado é EMPÍRICO — teste e anote.

In [ ]:
!mkdir -p /content/drive/MyDrive/TTS-ptbr-data/checkpoints/ && cp -r /content/output /content/drive/MyDrive/TTS-ptbr-data/checkpoints/qwen3tts_lora_v1
print('✅ salvo no Drive')
# Geração direta (sem LoRA-script) com a API do pacote — exemplo p/ teste de instruct:
# import torch; from qwen_tts import Qwen3TTSModel
# tts = Qwen3TTSModel.from_pretrained('Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice', device_map='cuda:0',
#                                     dtype=torch.bfloat16, attn_implementation='sdpa')
# wavs, sr = tts.generate_custom_voice(text='Caraca, que dia lindo no Rio!', language='Portuguese',
#                                      speaker='<preset>', instruct='fale com tom muito animado')